> **데이터셋 안내** — 이 노트북이 참조하는 HF 데이터셋은 공개 배포하지 않는다.
> AI Hub 원본에서 재생성하는 절차는 [docs/data/data-pipeline.md](../docs/data/data-pipeline.md)「가공 데이터셋은 배포하지 않는다 — 재현 경로」에 있다.

# 라벨 충돌·중복 제거

**동기.** 서로 다른 `documentId`인데 모델 입력(`invention_title + ipc_main + abstract + claims`)이 바이트 단위로 동일한 경우가 있다(공개/등록 문서, 가족·분할 출원 등). 이를 입력 문자열 동일성으로 검출해 세 부류를 제거한다.

1. **라벨 충돌** — 같은 입력에 라벨셋이 2개 이상. 어느 것이 옳은지 판정 불가 → 그룹 전원 제거. (42그룹 / 89문서)
2. **train 내 정확-중복** — 같은 입력·같은 라벨이 train에 여러 개 → train 사본 1개만 남기고 제거. (199문서)
3. **eval 정확-중복 누수** — 같은 입력·같은 라벨이 train과 eval 양쪽에 → train 사본 유지, **eval 사본 제거**. (val 20 + test 25 = 45문서)
4. **val+test 정확-중복** — train은 없고 val·test에 같은 입력·같은 라벨이 걸침 → **test 사본 유지, val 사본 제거**. (3그룹)

**재토큰화 불필요.** 토큰화본(`...-modernbert-tokenized`)은 `document_id`를 유지하고 원본과 **행 순서가 완전히 동일**하며, 토큰화는 행별 결정적이라 **행을 빼기만 하면** 재토큰화 결과와 같다. 따라서 원본·토큰본 **양쪽을 같은 `document_id` 집합으로 필터링**한다(느린 재토큰화 생략).

In [1]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
from collections import defaultdict, Counter

load_dotenv()
ROOT = Path(os.environ["DATA_ROOT"])
os.environ["HF_HOME"] = str(ROOT / ".hf_cache")
from datasets import load_dataset, DatasetDict

In [2]:
config = {
    "raw_ds": "ingyoun/patent-clean-text",
    "tokenized_repo": "ingyoun/patent-clean-text-modernbert-tokenized",
    "fields": ["invention_title", "ipc_main", "abstract", "claims"],  # 04_01 모델 입력 조합
    "out_path": ROOT / "output",
}
DRY_RUN = False   # 검토 후 False로 실행해 Hub 반영
OUT = config["out_path"]
SPLITS = ("train", "val", "test")

def build_input(ex):
    return " ".join(str(ex[f]) for f in config["fields"] if ex[f])

## 1. 검출 + 제거 정책

전 split을 모델 입력으로 그룹핑한 뒤 정책 적용. 정확-중복은 **train 사본을 우선 유지**(학습 데이터 보존·eval 정화)한다.

In [3]:
raw = {sp: load_dataset(config["raw_ds"], split=sp) for sp in SPLITS}
split_of = {}
grp = defaultdict(list)   # input -> [(split, document_id, frozenset(label_ids))]
for sp in SPLITS:
    for ex in raw[sp]:
        split_of[ex["document_id"]] = sp
        grp[build_input(ex)].append((sp, ex["document_id"], frozenset(ex["label_ids"])))

remove = set()
conflict_docs, train_dedup_docs, eval_leak_docs, valtest_dedup_docs = [], [], [], []
conflict_groups, valtest_groups = [], []

for t, g in grp.items():
    labelsets = {lb for _, _, lb in g}
    if len(labelsets) >= 2:                                  # (1) 라벨 충돌 — 전원 제거
        conflict_groups.append(g)
        for _, d, _ in g:
            remove.add(d); conflict_docs.append(d)
        continue
    if len(g) < 2:
        continue
    trains = [d for s, d, _ in g if s == "train"]            # 정확-중복(동일 입력·동일 라벨)
    if trains:                                               # train 포함 → train 1개 유지
        keep = trains[0]
        for s, d, _ in g:
            if d == keep: continue
            remove.add(d)
            (train_dedup_docs if s == "train" else eval_leak_docs).append(d)
    else:                                                    # train 없음
        tests = [d for s, d, _ in g if s == "test"]
        if tests:                                            # (4) val+test → test 1개 유지, 나머지(val) 제거
            keep = tests[0]
            valtest_groups.append(g)
            for s, d, _ in g:
                if d == keep: continue
                remove.add(d); valtest_dedup_docs.append(d)
        # test 없는 val-only 정확-중복(same-split)은 범위 밖(유지)

print(f"(1) 라벨 충돌         : {len(conflict_groups):>3} 그룹 → 제거 {len(conflict_docs)}")
print(f"(2) train 내 정확-중복 : 제거 {len(train_dedup_docs)}")
print(f"(3) eval 정확-중복 누수 : 제거 {len(eval_leak_docs)}  (val {sum(split_of[d]=='val' for d in eval_leak_docs)} + test {sum(split_of[d]=='test' for d in eval_leak_docs)})")
print(f"(4) val+test 정확-중복 : {len(valtest_groups)} 그룹 → test 유지·제거 {len(valtest_dedup_docs)}  (val {sum(split_of[d]=='val' for d in valtest_dedup_docs)} + test {sum(split_of[d]=='test' for d in valtest_dedup_docs)})")
print(f"\n총 제거 documentId: {len(remove)}   split 분포: {dict(Counter(split_of[d] for d in remove))}")

(1) 라벨 충돌         :  42 그룹 → 제거 89
(2) train 내 정확-중복 : 제거 199
(3) eval 정확-중복 누수 : 제거 45  (val 20 + test 25)
(4) val+test 정확-중복 : 3 그룹 → test 유지·제거 3  (val 3 + test 0)

총 제거 documentId: 336   split 분포: {'train': 279, 'test': 27, 'val': 30}


In [4]:
# SSOT 저장
rec = {
    "criterion": "모델 입력(invention_title+ipc_main+abstract+claims, 빈 필드 skip) 동일성 기준 그룹핑",
    "policy": "라벨충돌=그룹전원제거 · train포함정확중복=train1개유지후나머지제거 · val+test정확중복=test1개유지후val제거 · val전용중복=유지",
    "n_removed_total": len(remove),
    "removed_split_dist": dict(Counter(split_of[d] for d in remove)),
    "conflict_docs": sorted(conflict_docs),
    "train_dedup_docs": sorted(train_dedup_docs),
    "eval_leak_docs": sorted(eval_leak_docs),
    "valtest_dedup_docs": sorted(valtest_dedup_docs),
    "conflict_groups": [[f"{s}:{d}:{sorted(lb)}" for s, d, lb in g] for g in conflict_groups],
    "valtest_groups": [[f"{s}:{d}" for s, d, _ in g] for g in valtest_groups],
}
(OUT / "label_conflict_docs.json").write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", OUT / "label_conflict_docs.json")

saved: C:\workspace\patent_disc\output\label_conflict_docs.json


In [5]:
print("\n충돌 그룹(입력 동일·라벨 상이):")
for g in sorted(conflict_groups, key=lambda x: x[0][1])[:5]:
    print("  " + " | ".join(f"{d}[{s}]={sorted(lb)}" for s, d, lb in g))


충돌 그룹(입력 동일·라벨 상이):
  jp2000063731b2[train]=[11, 12, 180] | jp2000063731a[train]=[179]
  kr20010044812a[train]=[15] | kr20010044812b1[train]=[31]
  kr20020023625a[train]=[15] | kr20020023625b1[val]=[24]
  kr20037007840a[train]=[15] | kr20037007840b1[train]=[19]
  kr20040001320a[train]=[15] | kr20040001320b1[train]=[18, 19]


## 2. 원본↔토큰본 정합 확인 → 양쪽 필터링 (재토큰화 없음)

토큰화본이 `document_id`를 갖고 원본과 행 순서가 같음을 확인한 뒤, 같은 제거 집합으로 양쪽을 필터링한다.

In [6]:
tok = {sp: load_dataset(config["tokenized_repo"], split=sp) for sp in SPLITS}
assert "document_id" in tok["train"].column_names, "토큰화본에 document_id 없음"
for sp in SPLITS:
    assert raw[sp]["document_id"] == tok[sp]["document_id"], f"{sp}: 원본↔토큰본 행 순서 불일치"
print("정합 확인: 토큰화본에 document_id 존재 · 원본과 행 순서 동일(3 split)")

정합 확인: 토큰화본에 document_id 존재 · 원본과 행 순서 동일(3 split)


In [7]:
clean_base = DatasetDict({sp: raw[sp].filter(lambda ex: ex["document_id"] not in remove) for sp in SPLITS})
clean_tok  = DatasetDict({sp: tok[sp].filter(lambda ex: ex["document_id"] not in remove) for sp in SPLITS})

print(f"{'split':6}{'before':>9}{'after':>9}{'removed':>9}")
for sp in SPLITS:
    print(f"{sp:6}{len(raw[sp]):>9,}{len(clean_base[sp]):>9,}{len(raw[sp])-len(clean_base[sp]):>9}")
    # 원본·토큰본 정화 결과가 동일 문서·동일 순서인지 검증
    assert clean_base[sp]["document_id"] == clean_tok[sp]["document_id"], f"{sp}: 정화 후 원본↔토큰본 불일치"

# 잔여 충돌 0 검증(정리된 base)
g2 = defaultdict(set)
for sp in SPLITS:
    for ex in clean_base[sp]:
        g2[build_input(ex)].add(frozenset(ex["label_ids"]))
assert sum(len(v) >= 2 for v in g2.values()) == 0, "잔여 충돌 존재"
print("verify: 정화 후 원본==토큰본(문서·순서) · 잔여 라벨 충돌 0")

split    before    after  removed
train   201,895  201,616      279
val      11,162   11,132       30
test     11,271   11,244       27
verify: 정화 후 원본==토큰본(문서·순서) · 잔여 라벨 충돌 0


In [ ]:
# push (DRY_RUN 게이트)
if DRY_RUN:
    print("[DRY_RUN] push 생략")
    print("  base :", config["raw_ds"], {sp: len(clean_base[sp]) for sp in SPLITS})
    print("  token:", config["tokenized_repo"], {sp: len(clean_tok[sp]) for sp in SPLITS})
else:
    clean_base.push_to_hub(config["raw_ds"])
    clean_tok.push_to_hub(config["tokenized_repo"])
    print("pushed:", config["raw_ds"], "·", config["tokenized_repo"])